<a href="https://colab.research.google.com/github/yuvraj-kumar-dev/LawgicAI-Chatbot/blob/main/Non_Instructional_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:


In [3]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.0 MB/s eta 0:00:00


### Load Custom data (Domain Specific) for NON INSTRUCTIONAL FINETUNING

In [4]:
import fitz

In [5]:
def extract_text(pdf_path):
  text_block = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_block.append(text)
  return text_block

In [6]:
pdf_texts = extract_text("/content/constitution_of_india.pdf")

In [7]:
pdf_texts

['भारत का संविधान \n[1  मई, 2026 को यथाविद्यमान] \n \n \n THE CONSTITUTION OF INDIA \n[As on 1st May, 2026] \n \n2026 \n \nभारत सरकार \nविवध और न्याय मंत्रालय \nविधायी विभाग \nGOVERNMENT OF INDIA \nMINISTRY OF LAW AND JUSTICE \nLEGISLATIVE DEPARTMENT',
 'PREFACE \n \nThis is the sixth pocket size edition of the Constitution of \nIndia in the diglot form. In this edition, the text of the Constitution \nof India has been brought up-to-date by incorporating therein all \nthe amendments up to the Constitution (One Hundred and Sixth \nAmendment) Act, 2023. The foot notes below the text indicate the \nConstitution Amendment Acts by which such amendments have \nbeen made. \nThe Constitution (One Hundredth Amendment) Act, 2015 \ncontaining details of acquired and transferred territories between \nthe Governments of India and Bangladesh has been provided in \nAppendix I. \nThe Constitution (Application to Jammu and Kashmir) \nOrder, 2019 and the declaration under article 370(3) of the \nConstit

In [8]:
pdf_texts[0]

'भारत का संविधान \n[1  मई, 2026 को यथाविद्यमान] \n \n \n THE CONSTITUTION OF INDIA \n[As on 1st May, 2026] \n \n2026 \n \nभारत सरकार \nविवध और न्याय मंत्रालय \nविधायी विभाग \nGOVERNMENT OF INDIA \nMINISTRY OF LAW AND JUSTICE \nLEGISLATIVE DEPARTMENT'

In [9]:
len(pdf_texts)

402

In [10]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [11]:
paragraphs = split_paragraphs(pdf_texts)

In [17]:
data = [{'text': p} for p in paragraphs]

In [20]:
print(data)

[{'text': 'भारत का संविधान \n[1  मई, 2026 को यथाविद्यमान]'}, {'text': 'THE CONSTITUTION OF INDIA \n[As on 1st May, 2026]'}, {'text': 'भारत सरकार \nविवध और न्याय मंत्रालय \nविधायी विभाग \nGOVERNMENT OF INDIA \nMINISTRY OF LAW AND JUSTICE \nLEGISLATIVE DEPARTMENT'}, {'text': 'This is the sixth pocket size edition of the Constitution of \nIndia in the diglot form. In this edition, the text of the Constitution \nof India has been brought up-to-date by incorporating therein all \nthe amendments up to the Constitution (One Hundred and Sixth \nAmendment) Act, 2023. The foot notes below the text indicate the \nConstitution Amendment Acts by which such amendments have \nbeen made. \nThe Constitution (One Hundredth Amendment) Act, 2015 \ncontaining details of acquired and transferred territories between \nthe Governments of India and Bangladesh has been provided in \nAppendix I. \nThe Constitution (Application to Jammu and Kashmir) \nOrder, 2019 and the declaration under article 370(3) of the \n

In [21]:
from datasets import Dataset, load_dataset

dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['text'],
    num_rows: 696
})

### Load Model for training

In [22]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [27]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [32]:
def tokenize_fn(example):
  tokens = tokenizer(example['text'], truncation=True, padding='max_length', max_length=512)
  tokens['label'] = tokens['input_ids'].copy()
  return tokens

In [33]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])

Map:   0%|          | 0/696 [00:00<?, ? examples/s]

In [34]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'label'],
    num_rows: 696
})

In [35]:
model = AutoModelForCausalLM.from_pretrained(model)

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [38]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./llama_constitution_domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [ ]:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized
# )


Here  (above) we are not specfiying anything means this is full fine-tuning